In [ ]:
import cv2
import urllib.request
import os
from deepface import DeepFace
#print(cv2.__version__)  # Imprime la versión de OpenCV
#print(cv2.__file__)  # Imprime la ruta de los clasificadores Haar

In [ ]:
# Foto validación
imagen_referencia = "test.jpg"  
print(imagen_referencia)

test.jpg


In [ ]:
#carga del clasificador Haar Cascade para detección de rostros
xml_url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml"
xml_path = "haarcascade_frontalface_default.xml"
if not os.path.exists(xml_path):
    urllib.request.urlretrieve(xml_url, xml_path)

face_cascade = cv2.CascadeClassifier(xml_path)

In [ ]:
#proceso de captura de video
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: No se pudo abrir la cámara.")
else:
    frame_skip = 15
    contador = 0
    estado = "Esperando..."
    color_estado = (0, 0, 255)
    ultima_distancia = 0.0
    
    print("📸 Cámara abierta. Presiona 'q' para salir.")
    print(f"🔍 Comparando con: {imagen_referencia}")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Detectar rostros
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        caras = face_cascade.detectMultiScale(gray, 
                                              scaleFactor=1.05, 
                                              minNeighbors=3, 
                                              minSize=(80, 80))

        if len(caras) > 0:
            (x, y, w, h) = caras[0]
            cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)

            if contador % frame_skip == 0:
                try:
                    # Recortar la cara para enviar a DeepFace
                    cara_recortada = frame[y:y+h, x:x+w]
                    cara_rgb = cv2.cvtColor(cara_recortada, cv2.COLOR_BGR2RGB)

                    # Verificar
                    resultado = DeepFace.verify(
                        img1_path=imagen_referencia,
                        img2_path=cara_rgb,
                        enforce_detection=False,
                        model_name='ArcFace',  # Modelo más preciso
                        threshold=0.5
                        
                    )
                    
                    ultima_distancia = resultado['distance']
                    
                    if resultado['verified']:
                        estado = f"COINCIDENCIA ({ultima_distancia:.3f})"
                        color_estado = (0, 255, 0)
                    else:
                        estado = f"No coincide ({ultima_distancia:.3f})"
                        color_estado = (0, 0, 255)
                        
                except Exception as e:
                    estado = "Error"
                    color_estado = (0, 0, 255)

            contador += 1
        else:
            estado = "Error al detectar rostro"
            color_estado = (0, 0, 255)

        # Mostrar estado
        cv2.putText(frame, estado, (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color_estado, 2)
        cv2.imshow("Verificación Facial con DeepFace", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("👋 Cámara cerrada.")

📸 Cámara abierta. Presiona 'q' para salir.
🔍 Comparando con: test.jpg
👋 Cámara cerrada.
